# lora_convert — FSDP checkpoint to loadable adapter

Each run saved `checkpoint-4000/pytorch_model_fsdp.bin`: the full 3,972 M-param transformer with the adapter fused in, not a loadable adapter. This notebook reproduces `convert_ckpt_to_hf_format.py` — rebuild the adapter config, load the fused state dict, extract the LoRA layers via `get_peft_model_state_dict`, write `pytorch_lora_weights.safetensors` — with two environment fixes:

1. **torchao removed.** peft's LoRA injection walks every dispatcher, including `dispatch_torchao`, which raises rather than returning False when torchao predates 0.16 (Colab ships 0.10). torchao is unused by the bf16 path; removing it lets injection proceed. Same fix the training notebook applies.
2. **CPU build, not meta.** The shipped script builds under `init_empty_weights()` then calls `add_adapter`; on OmniGen2's architecture the current peft cannot size the LoRA layers on meta tensors. peft cannot be pinned down instead — diffusers requires `peft>=0.17` to import. Building on a concrete CPU module gives every target projection real weights, so injection succeeds; `load_state_dict(assign=True)` then replaces every base and adapter tensor with the checkpoint's values, so the extracted adapter is identical to the script's intent. Verified end to end under peft 0.19.

Run on a high-RAM runtime: constructing the model on CPU plus the mmap'd 15.9 GB checkpoint needs more than a standard runtime's ~12 GB. The A100 runtime (83 GB system RAM) or a High-RAM CPU runtime is fine.

## 1. Repo, environment fixes

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's LoRA injection walks every dispatcher, including dispatch_torchao, which raises rather than
# returning False when torchao predates 0.16 (Colab ships 0.10). torchao is unused by this bf16 path;
# removing it makes is_torchao_available() return False so the dispatcher is skipped. Same fix the
# training notebook applies. Guarded so it is a no-op once torchao is already gone / new enough.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

sys.path.insert(0, REPO)
os.chdir(REPO)

# diffusers requires peft>=0.17 to import; the convert cell builds the model on CPU (not a meta
# device) so LoRA injection works under that peft. Report versions and confirm the imports resolve.
import peft, diffusers, transformers
print(f"peft {peft.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
assert tuple(int(x) for x in peft.__version__.split(".")[:2]) >= (0, 17), \
    "diffusers needs peft>=0.17; do not pin peft below it"
_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel;"
          "from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline;"
          "print('convert imports OK')")
_chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(_chk.stdout.strip() or _chk.stderr.strip()[-800:])
assert _chk.returncode == 0, "converter imports are broken"

installed.
torchao removed
peft 0.19.1 | diffusers 0.39.0 | transformers 4.51.3
convert imports OK


## 2. Drive, checkpoint inventory

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
EXP   = f"{DRIVE}/lora/experiments"
STEP  = 4000
RANKS = [4, 8, 16]

# the training runs wrote through a symlink at /content/OmniGen2/experiments; the converter
# path arguments below point straight at Drive, so no symlink is required here
for r in RANKS:
    cfg = f"{EXP}/lora_r{r}/lora_r{r}.yml"
    ckpt = f"{EXP}/lora_r{r}/checkpoint-{STEP}/pytorch_model_fsdp.bin"
    assert os.path.exists(cfg),  f"missing config: {cfg}"
    assert os.path.exists(ckpt), f"missing checkpoint: {ckpt}"
    sz = os.path.getsize(ckpt) / 1e9
    print(f"r{r}: config + checkpoint present ({sz:.2f} GB)")

Mounted at /content/drive
r4: config + checkpoint present (15.88 GB)
r8: config + checkpoint present (15.89 GB)
r16: config + checkpoint present (15.91 GB)


## 3. Convert — fused checkpoint to adapter

In [ ]:
# The shipped convert_ckpt_to_hf_format.py builds the transformer under init_empty_weights()
# (meta device) before add_adapter. With a current peft that path raises in _create_new_module.
# This reproduces the script's logic exactly, but constructs the module on CPU so the LoRA layers
# materialise normally. load_state_dict(assign=True) then replaces every base+adapter tensor with
# the values from the checkpoint, so the extracted adapter is identical to the script's intent.
import torch, gc
from omegaconf import OmegaConf
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline

TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]

def convert(rank):
    cfg  = f"{EXP}/lora_r{rank}/lora_r{rank}.yml"
    ckpt = f"{EXP}/lora_r{rank}/checkpoint-{STEP}/pytorch_model_fsdp.bin"
    save = f"{EXP}/lora_r{rank}/checkpoint-{STEP}/transformer_lora"

    conf = OmegaConf.load(cfg)
    assert conf.train.get("lora_ft", False), f"r{rank}: config is not a LoRA run"
    arch_opt = OmegaConf.to_object(conf.model.arch_opt)
    for k, v in arch_opt.items():
        if isinstance(v, list): arch_opt[k] = tuple(v)

    # real CPU module (no init_empty_weights) so peft can inject LoRA on concrete tensors
    transformer = OmniGen2Transformer2DModel(**arch_opt)
    transformer.add_adapter(LoraConfig(
        r=conf.train.lora_rank, lora_alpha=conf.train.lora_rank,
        lora_dropout=conf.train.lora_dropout, init_lora_weights="gaussian",
        target_modules=TARGET_MODULES,
    ))

    state_dict = torch.load(ckpt, mmap=True, weights_only=True)
    missing, unexpect = transformer.load_state_dict(state_dict, assign=True, strict=False)
    # base weights are expected in `missing` only if absent; here the fused checkpoint carries them.
    lora_missing = [k for k in missing if "lora_" in k]
    assert not lora_missing, f"r{rank}: adapter keys not filled from checkpoint: {lora_missing[:4]}"
    assert not unexpect,     f"r{rank}: unexpected keys in checkpoint: {unexpect[:4]}"

    adapter = get_peft_model_state_dict(transformer)
    OmniGen2Pipeline.save_lora_weights(save_directory=save, transformer_lora_layers=adapter)
    n = sum(v.numel() for v in adapter.values())
    print(f"r{rank}: {len(adapter)} adapter tensors, {n/1e6:.3f} M params -> {save}")

    del transformer, state_dict, adapter
    gc.collect()

for r in RANKS:
    print(f"\n===== converting r{r} =====")
    convert(r)
print("\nall conversions done")

/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



===== converting r4 =====
r4: 304 adapter tensors, 2.554 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r4/checkpoint-4000/transformer_lora

===== converting r8 =====
r8: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r8/checkpoint-4000/transformer_lora

===== converting r16 =====
r16: 304 adapter tensors, 10.214 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/lora_r16/checkpoint-4000/transformer_lora

all conversions done


## 4. Verify adapters

In [ ]:
# verify each adapter exists, is small, and reports the expected trainable-parameter count for its rank.
# LoRA params = 2 * rank * sum(in+out over the 4 target projections); the ratio must be exactly 2x from r4 to r8 to r16.
from safetensors.torch import load_file

prev = None
for r in RANKS:
    path = f"{EXP}/lora_r{r}/checkpoint-{STEP}/transformer_lora/pytorch_lora_weights.safetensors"
    assert os.path.exists(path), f"adapter not written: {path}"
    sd = load_file(path)
    n_params = sum(v.numel() for v in sd.values())
    n_tensors = len(sd)
    sz = os.path.getsize(path) / 1e6
    ratio = "" if prev is None else f"  ({n_params/prev:.2f}x prev)"
    print(f"r{r}: {n_tensors} tensors, {n_params/1e6:.3f} M params, {sz:.1f} MB{ratio}")
    prev = n_params

# spot-check dtype and key form on the last one
k = next(iter(sd))
print(f"\nexample key: {k}")
print(f"dtype: {sd[k].dtype}")

r4: 304 tensors, 2.554 M params, 10.2 MB
r8: 304 tensors, 5.107 M params, 20.5 MB  (2.00x prev)
r16: 304 tensors, 10.214 M params, 40.9 MB  (2.00x prev)

example key: transformer.context_refiner.0.attn.to_k.lora_A.weight
dtype: torch.float32
